In [1]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

from langchain_ollama import ChatOllama

In [2]:
embedding_model_name = "gemma-2-embed"
lang_model_name = "tiger-gemma2"

max_tokens = 1024

In [ ]:
llm = ChatOllama(
    model=lang_model_name,
    temperature=0.8,
    num_predict=max_tokens,
    num_gpu=-1,
    # config={"extra": "allow"}
    # extra_fields_behavior="allow",
)

# import os
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [6]:
from typing import Optional

from pydantic import BaseModel, Field


class Search(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    query: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    publish_year: Optional[int] = Field(None, description="Year video was published")

In [25]:
system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return only a list of database queries optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
structured_llm = llm.with_structured_output(Search)
# query_analyzer = {"question": RunnablePassthrough()} | prompt | structured_llm
query_analyzer = {"question": RunnablePassthrough()} | prompt | llm

In [34]:
# ResponseError: tiger-gemma2 does not support tools
response = query_analyzer.invoke("videos on RAG published in 2023")
response


AIMessage(content="```sql\nSELECT * FROM TutorialVideos WHERE subject = 'RAG' AND year = 2023;\n```", additional_kwargs={}, response_metadata={'model': 'tiger-gemma2', 'created_at': '2024-10-20T11:43:13.736528894Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 1061247099, 'load_duration': 41835345, 'prompt_eval_count': 92, 'prompt_eval_duration': 57565000, 'eval_count': 26, 'eval_duration': 806832000}, id='run-3f8e8b80-04b9-4668-bce8-c454f5a8158b-0', usage_metadata={'input_tokens': 92, 'output_tokens': 26, 'total_tokens': 118})

In [35]:
from pprint import pprint
pprint(response.content)

('```sql\n'
 "SELECT * FROM TutorialVideos WHERE subject = 'RAG' AND year = 2023;\n"
 '```')
